In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import os

#CẤU HÌNH ĐƯỜNG DẪN VÀ THAM SỐ
data_dir = '/content/drive/MyDrive/DATA Emoji RA0001'

IMG_WIDTH, IMG_HEIGHT = 200, 200
BATCH_SIZE = 64

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,       # Xoay nhẹ 15 độ
    width_shift_range=0.1,   # Dịch ngang nhẹ
    height_shift_range=0.1,  # Dịch dọc nhẹ
    zoom_range=0.15,         # Zoom cận mặt
    horizontal_flip=True,    # Lật gương (rất quan trọng với khuôn mặt)
    validation_split=0.2     # 20% để test
)

train_generator = datagen.flow_from_directory(
    data_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    data_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

NUM_CLASSES = len(train_generator.class_indices)
print(f"\n👥 Số lượng người cần nhận diện: {NUM_CLASSES}")
print(f"📌 Danh sách: {train_generator.class_indices}")

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_HEIGHT, IMG_WIDTH, 3))
base_model.trainable = False # Đóng băng lớp gốc

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

#BẮT ĐẦU HUẤN LUYỆN
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

checkpoint = ModelCheckpoint('/content/best_face_model.h5', monitor='val_accuracy', save_best_only=True)

EPOCHS = 50

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=[early_stop, checkpoint]
)
print("✅ ĐÃ TRAIN XONG VÀ LƯU MÔ HÌNH TẠI /content/best_face_model.h5")

Found 1789 images belonging to 39 classes.
Found 440 images belonging to 39 classes.

👥 Số lượng người cần nhận diện: 39
📌 Danh sách: {'Hà Nguyễn Minh Anh_31251022430': 0, 'Hoàng Đình Nguyên_31251027872': 1, 'Huỳnh Ngọc Vương_31251021464': 2, 'Huỳnh Nguyễn Phát Sơn_31251021998': 3, 'Huỳnh Tuyên Chương_31251021888': 4, 'Lê Duy Nhật_31251025698': 5, 'Lê Ngọc Bảo Trâm_31251025307': 6, 'Lê Nguyễn Anh Khoa_31251025754': 7, 'Lý Tuấn Đạt_31251021889': 8, 'Ngô Quân Hạo_31251024029': 9, 'Nguyễn Anh Đức_31251028367': 10, 'Nguyễn Châu Nguyệt Mẫn_31251025080': 11, 'Nguyễn Duy Quang_31251020812': 12, 'Nguyễn Huy Khanh_31251020227': 13, 'Nguyễn Nam Hồng Phúc_31251020111': 14, 'Nguyễn Tấn Khang_31251026880': 15, 'Nguyễn Thị Minh Thư - 31251022343': 16, 'Nguyễn Thị Minh Thư_31251022940': 17, 'Nguyễn Trung Thủy Trúc_31251021890': 18, 'Nguyễn Đăng Hải Đăng - 31241023855': 19, 'Nguyễn Đức Trung_31251021463': 20, 'Ninh Công 

/tmp/ipykernel_624/3705009623.py:46: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_HEIGHT, IMG_WIDTH, 3))


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 29s/step - accuracy: 0.2115 - loss: 3.2153 

28/28 ━━━━━━━━━━━━━━━━━━━━ 1072s 38s/step - accuracy: 0.3762 - loss: 2.5288 - val_accuracy: 0.7773 - val_loss: 1.1900
Epoch 2/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7335 - loss: 1.1012

28/28 ━━━━━━━━━━━━━━━━━━━━ 41s 1s/step - accuracy: 0.7613 - loss: 0.9623 - val_accuracy: 0.8409 - val_loss: 0.6796
Epoch 3/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8683 - loss: 0.5642

...

28/28 ━━━━━━━━━━━━━━━━━━━━ 40s 1s/step - accuracy: 0.9419 - loss: 0.1912 - val_accuracy: 0.9114 - val_loss: 0.2667
Epoch 9/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9525 - loss: 0.1598

28/28 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.9480 - loss: 0.1663 - val_accuracy: 0.9205 - val_loss: 0.2520
Epoch 10/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9501 - loss: 0.1694

28/28 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.9503 - loss: 0.1698 - val_accuracy: 0.9341 - val_loss: 0.2204
Epoch 11/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.9514 - loss: 0.1400 - val_accuracy: 0.9341 - val_loss: 0.2327
Epoch 12/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.9530 - loss: 0.1412 - val_accuracy: 0.9136 - val_loss: 0.2562
Epoch 13/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.9547 - loss: 0.1143 - val_accuracy: 0.9250 - val_loss: 0.2351
Epoch 14/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 38s 1s/step - accuracy: 0.9542 - loss: 0.1194 - val_accuracy: 0.9250 - val_loss: 0.2412
Epoch 15/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.9558 - loss: 0.1140 - val_accuracy: 0.9159 - val_loss: 0.2456
Epoch 16/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.9570 - loss: 0.1056 - val_accuracy: 0.9341 - val_loss: 0.2332
Epoch 17/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9636 - loss: 0.0869

28/28 ━━━━━━━━━━━━━━━━━━━━ 38s 1s/step - accuracy: 0.9609 - loss: 0.0963 - val_accuracy: 0.9386 - val_loss: 0.1866
Epoch 18/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 38s 1s/step - accuracy: 0.9625 - loss: 0.0929 - val_accuracy: 0.9273 - val_loss: 0.1902
Epoch 19/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9600 - loss: 0.0854

28/28 ━━━━━━━━━━━━━━━━━━━━ 40s 1s/step - accuracy: 0.9631 - loss: 0.0833 - val_accuracy: 0.9409 - val_loss: 0.1975
Epoch 20/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.9625 - loss: 0.0851 - val_accuracy: 0.9341 - val_loss: 0.1852
Epoch 21/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 38s 1s/step - accuracy: 0.9603 - loss: 0.1017 - val_accuracy: 0.9295 - val_loss: 0.2169
Epoch 22/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.9693 - loss: 0.0805 - val_accuracy: 0.9341 - val_loss: 0.2188
Epoch 23/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.9709 - loss: 0.0772 - val_accuracy: 0.9409 - val_loss: 0.1690
Epoch 24/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.9637 - loss: 0.0795 - val_accuracy: 0.9341 - val_loss: 0.2065
Epoch 25/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 38s 1s/step - accuracy: 0.9653 - loss: 0.0738 - val_accuracy: 0.9295 - val_loss: 0.2208
Epoch 26/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.9659 - loss: 0.0744 - val_accuracy: 0.9386 - val_loss: 0.1

In [4]:
!pip install pillow-heif -q

import ipywidgets as widgets
from IPython.display import display, clear_output
import io
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from PIL import Image
import pillow_heif
import matplotlib.pyplot as plt

# Kích hoạt bộ đọc HEIC
pillow_heif.register_heif_opener()

print("⏳ Đang khởi động AI, vui lòng đợi vài giây...")

try:
    # Load mô hình đã được tải lên Colab (Lưu ý: phải upload file này lên trước khi chạy)
    model = load_model('/content/best_face_model.h5')
except Exception as e:
    print(" KHÔNG TÌM THẤY MÔ HÌNH!")
    raise e

labels = {
    0: 'Hà Nguyễn Minh Anh_31251022430',
    1: 'Hoàng Đình Nguyên_31251027872',
    2: 'Huỳnh Ngọc Vương_31251021464',
    3: 'Huỳnh Nguyễn Phát Sơn_31251021998',
    4: 'Huỳnh Tuyên Chương_31251021888',
    5: 'Lê Duy Nhật_31251025698',
    6: 'Lê Ngọc Bảo Trâm_31251025307',
    7: 'Lê Nguyễn Anh Khoa_31251025754',
    8: 'Lý Tuấn Đạt_31251021889',
    9: 'Ngô Quân Hạo_31251024029',
    10: 'Nguyễn Anh Đức_31251028367',
    11: 'Nguyễn Châu Nguyệt Mẫn_31251025080',
    12: 'Nguyễn Duy Quang_31251020812',
    13: 'Nguyễn Huy Khanh_31251020227',
    14: 'Nguyễn Nam Hồng Phúc_31251020111',
    15: 'Nguyễn Tấn Khang_31251026880',
    16: 'Nguyễn Thị Minh Thư - 31251022343',
    17: 'Nguyễn Thị Minh Thư_31251022940',
    18: 'Nguyễn Trung Thủy Trúc_31251021890',
    19: 'Nguyễn Đăng Hải Đăng - 31241023855',
    20: 'Nguyễn Đức Trung_31251021463',
    21: 'Ninh Công Uy_31251025315',
    22: 'Nông Đình Gia Phong_31251025849',
    23: 'Phạm Hoàng Thái_31251021462',
    24: 'Phạm Xuân Đăng_31251025847',
    25: 'Trần Khắc Triệu_31251022531',
    26: 'Trần Minh Mẫn_31251025599',
    27: 'Trần Thiên Hạo_31251022601',
    28: 'Trần Tiến Đạt_31251022119',
    29: 'Trần Vĩ Khang_31251024909',
    30: 'Võ Lê Gia Huy_31241023619',
    31: 'Võ Lý Thanh Nhã_31251023642',
    32: 'Đặng Hiếu Thiên_312410125173',
    33: 'Đặng Nhật duy Thái_31251025729',
    34: 'Đặng Quốc Vĩnh_31251024120',
    35: 'Đặng Thế Vũ _ 31251025972',
    36: 'Đoàn Thị Quỳnh Như_31251026015',
    37: 'Đỗ Dương Quang_31251021297',
    38: 'Đỗ Minh Nam_31251027063'
}

# Giao diện nút upload ảnh
uploader = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='Quét khuôn mặt',
    button_style='success'
)
out = widgets.Output()

def on_upload(change):
    with out:
        clear_output()
        if not uploader.value:
            return

        # Xử lý lấy file từ nút upload (Tương thích nhiều phiên bản Colab)
        try:
            file_info = uploader.value[0]
            content = file_info.content
            name = file_info.name
        except (KeyError, TypeError):
            name = list(uploader.value.keys())[0]
            content = uploader.value[name]['content']

        print(f"⏳ Đang phân tích ảnh: {name}...\n")

        try:
            # Đọc ảnh trực tiếp từ bộ nhớ
            img = Image.open(io.BytesIO(content))
            img = img.convert('RGB')

            # Hiển thị ảnh ra màn hình
            plt.figure(figsize=(4, 4))
            plt.imshow(img)
            plt.axis('off')
            plt.show()


            img_resized = img.resize((200, 200))
            img_array = tf.keras.preprocessing.image.img_to_array(img_resized)
            img_array = np.expand_dims(img_array, axis=0)
            img_array /= 255.0

            # AI tiến hành dự đoán
            predictions = model.predict(img_array, verbose=0)
            predicted_class = np.argmax(predictions[0])
            confidence = np.max(predictions[0]) * 100

            # In kết quả
            print(f"👤 Kết quả nhận diện: {labels.get(predicted_class, 'Không có trong dữ liệu')}")
            print(f"🎯 Độ tự tin: {confidence:.2f}%\n" + "="*40)

        except Exception as e:
            print(f"❌ Có lỗi xảy ra khi đọc bức ảnh này: {e}")

uploader.observe(on_upload, names='value')

print("✅ Đã tải xong AI! Sẵn sàng nhận diện.")
print("=== 📸 HỆ THỐNG DỰ ĐOÁN KHUÔN MẶT 📸 ===")
display(uploader, out)

⏳ Đang khởi động AI, vui lòng đợi vài giây...


✅ Đã tải xong AI! Sẵn sàng nhận diện.
=== 📸 HỆ THỐNG DỰ ĐOÁN KHUÔN MẶT 📸 ===


FileUpload(value={}, accept='image/*', button_style='success', description='Quét khuôn mặt')

Output()